In [1]:
import torch
import torch.nn as nn
import numpy as np
import tensorflow as tf
print("PyTorch:", torch.__version__)
print("TensorFlow:", tf.__version__)
print("All ready!")

PyTorch: 2.12.1+cpu
TensorFlow: 2.21.0
All ready!


In [2]:
# Recreate the CNN architecture
class ECG_CNN(nn.Module):
    def __init__(self, num_classes=4):
        super(ECG_CNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=5, padding=2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.bn1 = nn.BatchNorm1d(32)
        self.bn2 = nn.BatchNorm1d(64)
        self.bn3 = nn.BatchNorm1d(128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(128 * 22, 256)
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, num_classes)
    
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

# Load saved model
model = ECG_CNN(num_classes=4)
model.load_state_dict(torch.load(
    r'C:\Users\vaish\Desktop\ml-journey\week4\ecg_cnn_model.pth',
    map_location='cpu'
))
model.eval()
print("Model loaded successfully!")

# Test with dummy input
dummy = torch.randn(1, 1, 180)
with torch.no_grad():
    output = model(dummy)
print("Output shape:", output.shape)
print("Model ready for conversion!")

Model loaded successfully!
Output shape: torch.Size([1, 4])
Model ready for conversion!


In [3]:
import subprocess
import os

# Step 1 — Export PyTorch model to ONNX
dummy_input = torch.randn(1, 1, 180)
onnx_path = r'C:\Users\vaish\Desktop\ml-journey\week4\ecg_cnn.onnx'

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    input_names=['ecg_input'],
    output_names=['class_output'],
    dynamic_axes={
        'ecg_input': {0: 'batch_size'},
        'class_output': {0: 'batch_size'}
    }
)

print("✓ Step 1: PyTorch → ONNX done!")
print(f"  Saved to: {onnx_path}")
print(f"  File size: {os.path.getsize(onnx_path)/1024:.1f} KB")

C:\Users\vaish\AppData\Local\Temp\ipykernel_28164\385089731.py:8: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0623 18:02:04.953000 28164 site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `ECG_CNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ECG_CNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


C:\Users\vaish\AppData\Local\Programs\Python\Python313\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "C:\Users\vaish\AppData\Local\Programs\Python\Python313\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "C:\Users\vaish\AppData\Local\Programs\Python\Python313\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "C:\Users\v

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✓ Step 1: PyTorch → ONNX done!
  Saved to: C:\Users\vaish\Desktop\ml-journey\week4\ecg_cnn.onnx
  File size: 19.9 KB


In [4]:
import tensorflow as tf
import numpy as np

# Step 1 — Create equivalent TensorFlow model
def create_tf_model():
    inputs = tf.keras.Input(shape=(180, 1))
    
    x = tf.keras.layers.Conv1D(32, 5, padding='same', activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    
    x = tf.keras.layers.Conv1D(64, 5, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    
    x = tf.keras.layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(4, activation='softmax')(x)
    
    model_tf = tf.keras.Model(inputs, outputs)
    return model_tf

tf_model = create_tf_model()
tf_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
tf_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 180, 1)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d (Conv1D)                      │ (None, 180, 32)             │             192 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 180, 32)             │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d (MaxPooling1D)         │ (None, 90, 32)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ (None, 90, 64)              │          10,304 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 90, 64)              │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_1 (MaxPooling1D)       │ (None, 45, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_2 (Conv1D)                    │ (None, 45, 128)             │          24,704 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 45, 128)             │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_2 (MaxPooling1D)       │ (None, 22, 128)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 2816)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         721,152 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │          16,448 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 4)                   │             260 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 773,956 (2.95 MB)

 Trainable params: 773,508 (2.95 MB)

 Non-trainable params: 448 (1.75 KB)

In [5]:
# Prepare data for TensorFlow
# TF expects (batch, length, channels) — opposite of PyTorch

# Load the beats data again
import wfdb
from collections import Counter
from sklearn.utils import resample
from sklearn.model_selection import train_test_split

records = ['100', '101', '103', '105', '106', '108', '109', '111', '112', '113']

def extract_beats_multirecord(record_ids, window_size=180):
    all_beats = []
    all_labels = []
    valid_symbols = {'N': 0, 'A': 1, 'V': 2, 'L': 3}
    
    for rec_id in record_ids:
        try:
            record = wfdb.rdrecord(rec_id, pn_dir='mitdb')
            annotation = wfdb.rdann(rec_id, 'atr', pn_dir='mitdb')
            ecg_signal = record.p_signal[:, 0]
            for i, sample in enumerate(annotation.sample):
                symbol = annotation.symbol[i]
                if symbol not in valid_symbols:
                    continue
                start = sample - window_size // 2
                end = sample + window_size // 2
                if start < 0 or end > len(ecg_signal):
                    continue
                beat = ecg_signal[start:end]
                min_val, max_val = beat.min(), beat.max()
                if max_val - min_val != 0:
                    beat = (beat - min_val) / (max_val - min_val)
                all_beats.append(beat)
                all_labels.append(valid_symbols[symbol])
        except:
            pass
    return np.array(all_beats), np.array(all_labels)

print("Loading data...")
beats, labels = extract_beats_multirecord(records)

# Balance dataset
target_count = 500
balanced_beats = []
balanced_labels = []
for label in range(4):
    idx = np.where(labels == label)[0]
    b = beats[idx]
    l = labels[idx]
    b, l = resample(b, l, n_samples=target_count,
                    replace=len(b) < target_count, random_state=42)
    balanced_beats.append(b)
    balanced_labels.append(l)

X = np.vstack(balanced_beats)
y = np.hstack(balanced_labels)

# Reshape for TensorFlow — (samples, length, channels)
X = X.reshape(-1, 180, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print("Data ready!")

Loading data...
X_train shape: (1600, 180, 1)
X_test shape: (400, 180, 1)
Data ready!


In [6]:
# Train TensorFlow model
history = tf_model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

# Evaluate
test_loss, test_acc = tf_model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 6s 38ms/step - accuracy: 0.7625 - loss: 0.7500 - val_accuracy: 0.2850 - val_loss: 1.3365
Epoch 2/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.8825 - loss: 0.3324 - val_accuracy: 0.4375 - val_loss: 1.2845
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.9156 - loss: 0.2329 - val_accuracy: 0.4300 - val_loss: 1.2422
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.9406 - loss: 0.1580 - val_accuracy: 0.4275 - val_loss: 1.1464
Epoch 5/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - accuracy: 0.9531 - loss: 0.1401 - val_accuracy: 0.4650 - val_loss: 1.2514
Epoch 6/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.9619 - loss: 0.1200 - val_accuracy: 0.7650 - val_loss: 0.6521
Epoch 7/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9712 - loss: 0.0882 - val_accuracy: 0.7325 - val_loss: 0.6341
Epoch 8/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.9719 - loss: 0.0809 - val_accuracy: 0.8700 - v

In [8]:
# Save the TF model correctly
tf_model.export('ecg_cnn_tf_saved_model')
print("✓ TensorFlow model exported")

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_saved_model('ecg_cnn_tf_saved_model')
tflite_model = converter.convert()

# Save TFLite model
tflite_path = 'ecg_cnn_model.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

import os
tflite_size = os.path.getsize(tflite_path)
print(f"✓ TFLite model saved!")
print(f"TFLite model size: {tflite_size/1024:.1f} KB")
print(f"TFLite model size: {tflite_size/1024/1024:.2f} MB")

INFO:tensorflow:Assets written to: ecg_cnn_tf_saved_model\assets


INFO:tensorflow:Assets written to: ecg_cnn_tf_saved_model\assets


Saved artifact at 'ecg_cnn_tf_saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 180, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  1929073842064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1929650232912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1929650233104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1929650233296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1929650237520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1929650232720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1929650235792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1929650237328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1929650240016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1929650240784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1929650225232: TensorSpec(shape=(

In [9]:
# Quantize to int8 — makes model smaller and faster on hardware

def representative_dataset():
    for i in range(100):
        yield [X_train[i:i+1].astype(np.float32)]

# Convert with int8 quantization
converter_quant = tf.lite.TFLiteConverter.from_saved_model('ecg_cnn_tf_saved_model')
converter_quant.optimizations = [tf.lite.Optimize.DEFAULT]
converter_quant.representative_dataset = representative_dataset
converter_quant.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_quant.inference_input_type = tf.float32
converter_quant.inference_output_type = tf.float32

tflite_quant_model = converter_quant.convert()

# Save quantized model
quant_path = 'ecg_cnn_model_quantized.tflite'
with open(quant_path, 'wb') as f:
    f.write(tflite_quant_model)

original_size = os.path.getsize(tflite_path)
quant_size = os.path.getsize(quant_path)

print(f"✓ Quantized model saved!")
print(f"Original TFLite size : {original_size/1024:.1f} KB")
print(f"Quantized size       : {quant_size/1024:.1f} KB")
print(f"Size reduction       : {(1 - quant_size/original_size)*100:.1f}%")

✓ Quantized model saved!
Original TFLite size : 3034.3 KB
Quantized size       : 784.5 KB
Size reduction       : 74.1%


In [10]:
# Test quantized model accuracy

interpreter = tf.lite.Interpreter(model_path=quant_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input shape:", input_details[0]['shape'])
print("Output shape:", output_details[0]['shape'])

# Run inference on test set
y_pred_quant = []

for i in range(len(X_test)):
    input_data = X_test[i:i+1].astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    y_pred_quant.append(np.argmax(output))

y_pred_quant = np.array(y_pred_quant)

from sklearn.metrics import accuracy_score, f1_score, classification_report
acc = accuracy_score(y_test, y_pred_quant)
f1 = f1_score(y_test, y_pred_quant, average='weighted')

print(f"\nQuantized Model Results:")
print(f"Accuracy : {acc:.4f}")
print(f"F1 Score : {f1:.4f}")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred_quant,
      target_names=['Normal', 'Atrial', 'Ventricular', 'Left BB']))

Input shape: [  1 180   1]
Output shape: [1 4]

Quantized Model Results:
Accuracy : 0.8950
F1 Score : 0.8915

Detailed Report:
              precision    recall  f1-score   support

      Normal       0.71      1.00      0.83       100
      Atrial       1.00      0.61      0.76       100
 Ventricular       0.98      1.00      0.99       100
     Left BB       1.00      0.97      0.98       100

    accuracy                           0.90       400
   macro avg       0.92      0.90      0.89       400
weighted avg       0.92      0.90      0.89       400



C:\Users\vaish\AppData\Local\Programs\Python\Python313\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [11]:
print("=== Week 6 Summary ===")
print("TensorFlow model accuracy  : 99.50%")
print("TFLite model size          : 3034 KB")
print("Quantized model size       : 784 KB")
print("Size reduction             : 74.1%")
print("Quantized model accuracy   : 89.50%")
print("Quantized F1 score         : 89.15%")
print("\nModel ready for Raspberry Pi deployment!")
print("Next: Week 7 — Deploy on Raspberry Pi")

=== Week 6 Summary ===
TensorFlow model accuracy  : 99.50%
TFLite model size          : 3034 KB
Quantized model size       : 784 KB
Size reduction             : 74.1%
Quantized model accuracy   : 89.50%
Quantized F1 score         : 89.15%

Model ready for Raspberry Pi deployment!
Next: Week 7 — Deploy on Raspberry Pi
